In [207]:
from pymatgen.core import Lattice, Structure
import pandas as pd
import numpy as np
import plotly as pt
import seaborn as sns
#!pip install pymatgen
#!pip install mp_api
import requests
import json
import matplotlib.pyplot as plt
import re

In [208]:
from matminer.featurizers.composition import ElementProperty
from matminer.featurizers.composition import ValenceOrbital
from pymatgen.core.composition import Composition

In [ ]:
from mp_api.client import MPRester
API_KEY = ""
mpr = MPRester(API_KEY)

# Initialization


In [377]:
import joblib

title="Log_rate"  #'Bandgap  #Log_rate
scaler = joblib.load(f"{title}_scaler.joblib")
selector = joblib.load(f"{title}_selector.joblib")
cat_boost_model = joblib.load(f"{title}_regression_model.joblib")

print("Pipeline reloaded!")

Pipeline reloaded!


In [378]:
#Bandgap

avg_calc_temp = 1267.106509
avg_calc_time = 22
avg_surf_area = 9.371599670116556	
min_wl = 200

#****************************
#Log_rate
if(title=="Log_rate"):
    avg_calc_temp = 1296.237241
    avg_calc_time = 19.567693	
    avg_surf_area = 10.37065568513124	
    min_wl = 200

ionic_radii={"H+":0.02}


In [379]:
candidate_cif_folder = "generated_cifs"

from pathlib import Path
#from pymatgen.core import Structure

# Define the directory path
folder_path = Path(candidate_cif_folder)

data_rows = []
# Iterate over all files ending in .cif
for file_path in folder_path.glob("*.cif"):
    #print(f"Processing: {file_path.name}")
    
    # Example: Integration with your pymatgen code
    structure = Structure.from_file(file_path)
    formula = structure.composition.formula
    #print(f"Formula: {structure.composition.formula}")

    row_data = {
            "Formula": formula,
            "Composition": structure.composition,
            "Path": file_path
        }
    data_rows.append(row_data)

df_candidates = pd.DataFrame(data_rows)

if(title=='Log_rate'):
    df_candidates = pd.read_excel(f"{candidate_cif_folder}/candidates_after_inference_Bandgap.xlsx")
df_candidates

,Unnamed: 0,Formula,Composition,Path,avg s valence electrons,avg p valence electrons,avg d valence electrons,avg f valence electrons,frac s valence electrons,frac p valence electrons,frac d valence electrons,frac f valence electrons,Bandgap_predicted
0,0,Sr3 Ca1 Zn1 O5,Sr3 Ca1 Zn1 O5,generated_cifs\gen_10_Sr3CaZnO5.cif,2.000000,2.000000,1.000000,0.0,0.400000,0.400000,0.200000,0.000000,2.610747
1,1,Ca3 Nb1 O3,Ca3 Nb1 O3,generated_cifs\gen_11_Ca3NbO3.cif,1.857143,1.714286,0.571429,0.0,0.448276,0.413793,0.137931,0.000000,2.125691
2,2,Sr7 Ca1 O2,Sr7 Ca1 O2,generated_cifs\gen_12_Sr7CaO2.cif,2.000000,0.800000,0.000000,0.0,0.714286,0.285714,0.000000,0.000000,2.915892
3,3,Ca6 P2 O2,Ca6 P2 O2,generated_cifs\gen_13_Ca3PO.cif,2.000000,1.400000,0.000000,0.0,0.588235,0.411765,0.000000,0.000000,2.840694
4,4,Ca2 O2,Ca2 O2,generated_cifs\gen_14_CaO.cif,2.000000,2.000000,0.000000,0.0,0.500000,0.500000,0.000000,0.000000,3.424377
5,5,Ca1 Hg1 O2,Ca1 Hg1 O2,generated_cifs\gen_15_CaHgO2.cif,2.000000,2.000000,2.500000,3.5,0.200000,0.200000,0.250000,0.350000,2.675027
6,6,Ca4 O2,Ca4 O2,generated_cifs\gen_16_Ca2O.cif,2.000000,1.333333,0.000000,0.0,0.600000,0.400000,0.000000,0.000000,2.768946
7,7,Ca5 O3,Ca5 O3,generated_cifs\gen_17_Ca5O3.cif,2.000000,1.500000,0.000000,0.0,0.571429,0.428571,0.000000,0.000000,2.911306
8,8,Sr6 Ca1 Nb2 Zn1 O6,Sr6 Ca1 Nb2 Zn1 O6,generated_cifs\gen_18_Sr6CaNb2ZnO6.cif,1.875000,1.500000,1.125000,0.0,0.416667,0.333333,0.250000,0.000000,2.009304
9,9,Sr5 Nb3 O7,Sr5 Nb3 O7,generated_cifs\gen_19_Sr5Nb3O7.cif,1.800000,1.866667,0.800000,0.0,0.402985,0.417910,0.179104,0.000000,2.297273


In [380]:
print(df_candidates['Formula'].to_string(index=False))

    Sr3 Ca1 Zn1 O5
        Ca3 Nb1 O3
        Sr7 Ca1 O2
         Ca6 P2 O2
            Ca2 O2
        Ca1 Hg1 O2
            Ca4 O2
            Ca5 O3
Sr6 Ca1 Nb2 Zn1 O6
        Sr5 Nb3 O7
    Sr6 Ca1 Pb1 O2
Sr2 Ca1 Nb2 Br1 O6
        Sr1 Ca1 O2
            Ca4 O2
            Ca3 O1
    Sr1 Ca6 Nb2 O7
        Sr4 Ca2 O6
        Ca5 Fe1 O6
        Ca3 Bi1 O1
            Sr2 O2
            Ca4 O4
            Ca2 O2
    Sr2 Ca1 Nb1 O6
        Sr1 Nb2 O5
     Sr3 Ca3 O5 F1
   Sr12 Ca4 Zn1 O3
         Ca3 O2 F1
     La1 Nb2 H2 O7
            Ca4 O4
            Ca2 O2
            Ca2 O2
            Ca2 O2


In [381]:
vo_feat = ValenceOrbital()

In [382]:
if(title=='Bandgap'):
    df_candidates = vo_feat.featurize_dataframe(df_candidates, col_id='Composition')

In [383]:
#electron = ElementProperty()
#df_candidates = electron.featurize_dataframe(df_candidates, col_id='Composition')

In [384]:
df_candidates.columns

Index(['Unnamed: 0', 'Formula', 'Composition', 'Path',
       'avg s valence electrons', 'avg p valence electrons',
       'avg d valence electrons', 'avg f valence electrons',
       'frac s valence electrons', 'frac p valence electrons',
       'frac d valence electrons', 'frac f valence electrons',
       'Bandgap_predicted'],
      dtype='object')

In [385]:
from pymatgen.core.periodic_table import Element

#calculate malliken electronegativity of an element
def get_mulliken_en(element_symbol):
    el = Element(element_symbol)
    IE = el.ionization_energies[0]  # First ionization energy in eV
    EA = el.electron_affinity       # Electron affinity in eV

    if IE is None or EA is None:
        return None

    return (IE + EA) / 2

def calc_average_electronegativity(formula):
  # Example: For Fe2O3
  comp = Composition(formula)
  # Weighted mean electronegativity
  total_atoms = comp.num_atoms
  mean_en = sum(
    comp[el] / total_atoms * get_mulliken_en(el)
    for el in comp.elements)
  return mean_en

In [386]:
calc_average_electronegativity("Ti O2")

6.177000159666666

In [387]:
def split_formula(formula):
  output={}
  elements_with_indexes = formula.split()
  for el in elements_with_indexes:
    #match = re.match(r"([A-Za-z]+)(\d+)$", el)
    match = re.match(r"([A-Za-z]+)(\d+(?:\.\d+)?)$", el)
    if match:
      output[match.group(1)]=float(match.group(2))
    else:
      output[ el]=float(1)
  return output

def get_valence_electrons_number(hill_fomula):
  split = split_formula(hill_fomula)
  print(split)
  v = split.get("O")
  if v == None:
    return 0
  else:
    return 2*v

In [388]:
get_valence_electrons_number("Ti O2")

{'Ti': 1.0, 'O': 2.0}


4.0

In [389]:
elements_oxidation_states = {
"Pu":4,
"H":1,
"Li":1,
"Be":2,
"B":3,
"C":4,
"N":-3,
"O":-2,
"F":-1,
"Na":1,
"Mg":2,
"Al":3,
"Si":4,
"P":5,
"S":-2,
"Cl":-1,
"K":1,
"Ca":2,
"Sc":3,
"Ti":4,
"V":5,
"Cr":6,
"Mn":2,
"Fe":2,
"Co":2,
"Ni":2,
"Cu":2,
"Zn":2,
"Ga":3,
"Ge":4,
"As":5,
"Se":6,
"Br":-1,
"Rb":1,
"Sr":2,
"Y":3,
"Zr":4,
"Nb":5,
"Mo":6,
"Tc":7,
"Ru":4,
"Rh":3,
"Pd":2,
"Ag":1,
"Cd":2,
"In":3,
"Sn":4,
"Sb":5,
"Te":6,
"I":-1,
"Cs":1,
"Ba":2,
"La":3,
"Ce":4,
"Pr":3,
"Nd":3,
"Pm":3,
"Sm":3,
"Eu":3,
"Gd":3,
"Tb":3,
"Dy":3,
"Ho":3,
"Er":3,
"Tm":3,
"Yb":3,
"Lu":3,
"Hf":4,
"Ta":5,
"W":6,
"Re":7,
"Os":8,
"Ir":4,
"Pt":4,
"Au":3,
"Hg":2,
"Tl":1,
"Pb":2,
"Bi":3,
"Po":6,
"At":7,
"Th":4,
"Pa":5,
"U":6,
"Np":7,
"Xe":0,
}

In [390]:
from pymatgen.core import Structure
from pymatgen.analysis.local_env import ValenceIonicRadiusEvaluator
from pymatgen.core.periodic_table import Species

unresolved_compunds = []

def get_packing_fraction_from_formula_and_cell_volume(hill_formula, V, Z):
    if V==0 or np.isnan(V) or Z==0 or np.isnan(Z):
        return np.nan
    print("---------------------------------------------")
    print("Enter!")
    print(hill_formula)
    print(V)
    print(Z)
    comp = Composition(hill_formula)

    #oxi_guesses = comp.oxi_state_guesses()
    #if(len(oxi_guesses)==0):
    #   oxi_guesses = difficult_compunds_oxidation_states.get(hill_formula)
    #  if(oxi_guesses==None):
    #    unresolved_compunds.append(hill_formula)
    #    return np.nan
    #else:
    #  oxi_guesses = oxi_guesses[0]

    V_ions_formula=0
    for el, amt in comp.items():
        symbol = el.symbol
        #el_oxidation_state = oxi_guesses[symbol]
        el_oxidation_state=3
        if symbol in elements_oxidation_states:
          el_oxidation_state = elements_oxidation_states[symbol]
        print("Element:", symbol)
        print("Element ox state:", el_oxidation_state)
        specie = Species(symbol,el_oxidation_state)
        r = specie.ionic_radius
        if(r==np.nan or r==None):
          ion_formula = symbol
          if(el_oxidation_state!=0 and el_oxidation_state!=1 and el_oxidation_state!=-1):
             ion_formula =  ion_formula+str(el_oxidation_state)
          if(el_oxidation_state>0):
            ion_formula =  ion_formula+str("+")
          if(el_oxidation_state<0):
            ion_formula =  ion_formula+str("-")
          print("Local ionic radii table request for ",ion_formula)
          r= ionic_radii.get(ion_formula)
          if(r==None):
            r=0
        print("r: ", r)
        V_ion = (4/3) * np.pi * r**3
        V_ions_formula += (V_ion*amt)
        print(el," ",amt)

    packing_fraction= Z*V_ions_formula/V
    return packing_fraction
    print("Output!")

In [391]:
get_packing_fraction_from_formula_and_cell_volume("Ti O2", 1, 1)

---------------------------------------------
Enter!
Ti O2
1
1
Element: Ti
Element ox state: 4
r:  0.745 ang
Ti   1.0
Element: O
Element ox state: -2
r:  1.26 ang
O   2.0


18.490348835521182

In [392]:
def count_oxigen(formula):
  if formula is None:
    return 0
  print(formula)
  comp = Composition(formula)

  # Get number of oxygen atoms
  oxygen_count = comp.get_el_amt_dict().get("O", 0)
  print(oxygen_count)
  return oxygen_count

In [393]:
#from matminer.featurizers.composition import ElementProperty
#ep_feat = ElementProperty.from_preset(preset_name="magpie")

#comp = Composition("Nd3Lu2Tl1O4")
#feature_values = ep_feat.featurize(comp)

#feature_names = ep_feat.feature_labels()
#composition_features = dict(zip(feature_names, feature_values))

#print(f"Computed {len(composition_features)} features for single composition.")
#print("Example feature (Mulliken EN):", composition_features.get("MagpieData minimum Electronegativity"))
#print(feature_names)

comp = Composition("Nd3Lu2Tl1O4")
feature_names = vo_feat.feature_labels()
print(feature_names)
feature_values = vo_feat.featurize(comp)
print(feature_values)

['avg s valence electrons', 'avg p valence electrons', 'avg d valence electrons', 'avg f valence electrons', 'frac s valence electrons', 'frac p valence electrons', 'frac d valence electrons', 'frac f valence electrons']
[np.float64(2.0), np.float64(1.7), np.float64(1.2), np.float64(5.4), np.float64(0.1941747572815534), np.float64(0.16504854368932037), np.float64(0.11650485436893203), np.float64(0.5242718446601942)]


In [394]:
def predict_candidate_from_cif(cif_path, bandgap, load_cif_from_Data = False,d_A= 0):
    #if(bandgap==None or np.isnan(bandgap) or bandgap<=0):
    #    return None
    print("Predicting for: ", cif_path)
    cif_path = str(cif_path)
    #print("Predicting for: ", cif_path)
    #try:
    candidate_feature_dict={}
    structure = None

    if(cif_path == 'M_MP491'):
        c=0
    
    try:
        if cif_path.startswith("mp-"):
            structure = mpr.get_structure_by_material_id(cif_path)
            print("loading cif from web")
        elif(load_cif_from_Data):
            structure = Structure.from_file(f"Data/CIF/{cif_path}.cif")
            print("loading cif locally from DATA")
        else:
            print("loading cif locally")
            structure = Structure.from_file(cif_path)
            print("loadied cif locally")
        if structure is None:
            print("Could not read structure from CIF file.")
            return None
    except:
        return np.nan
    print("Structure is loaded")
    composition = structure.composition
    formula = composition.formula
    print(formula) 

    candidate_feature_dict['CalcT(K)'] = avg_calc_temp
    candidate_feature_dict['Calc time (h)'] = avg_calc_time
    candidate_feature_dict['Promoter, w%'] = 0
    candidate_feature_dict['Surface area, m2/g'] = avg_surf_area
    candidate_feature_dict['Alcohol, %'] = 1
    candidate_feature_dict['Power, W'] = 1
    candidate_feature_dict['Wave length min, nm'] = min_wl
    candidate_feature_dict['Promotion method_PD'] = True
    candidate_feature_dict['Promoter_Pt'] = True
    candidate_feature_dict['Promoter_Rh'] = False
    candidate_feature_dict['Combined feature'] = 1*1*1*1
    if(np.isnan(d_A)):
        print("d,A = NaN")
        candidate_feature_dict['d,A'] = 0
    else:
        candidate_feature_dict['d,A'] = d_A
    candidate_feature_dict['Temperature, K'] = 298
    candidate_feature_dict['Nitrogen'] = False
    candidate_feature_dict['Prep Method_SSR'] = True

    #avg_d_valence_electrons = df_candidates.loc[df_candidates["Formula"] == formula, "avg d valence electrons"].item()
    #print("avg d elect: ",  avg_d_valence_electrons)
    #candidate_feature_dict['avg d valence electrons'] =  avg_d_valence_electrons

    #candidate_feature_dict['avg s valence electrons'] =  df_candidates.loc[df_candidates["Formula"] == formula, "avg s valence electrons"].item()
    #candidate_feature_dict['avg p valence electrons'] =  df_candidates.loc[df_candidates["Formula"] == formula, "avg p valence electrons"].item()
    #candidate_feature_dict['avg f valence electrons'] =  df_candidates.loc[df_candidates["Formula"] == formula, "avg f valence electrons"].item()

    #candidate_feature_dict['frac s valence electrons'] =  df_candidates.loc[df_candidates["Formula"] == formula, "frac s valence electrons"].item()
    #candidate_feature_dict['frac p valence electrons'] =  df_candidates.loc[df_candidates["Formula"] == formula, "frac p valence electrons"].item()
    #candidate_feature_dict['frac f valence electrons'] =  df_candidates.loc[df_candidates["Formula"] == formula, "frac f valence electrons"].item()

    feature_names = vo_feat.feature_labels()
    print(feature_names)
    feature_values = vo_feat.featurize(composition)
    print(feature_values)
    candidate_vo_feature_dict={}
    candidate_vo_feature_dict.update(zip(feature_names, feature_values))

    for f in feature_names:
        if f in candidate_vo_feature_dict:
            candidate_feature_dict[f]=candidate_vo_feature_dict[f]
        else:
            print(" error adding feature ", f, " to candidate feature dict")
            raise ValueError(" error adding feature ", f, " to candidate feature dict")

    #candidate_feature_dict['Bandgap, eV'] = avg_bandgap
    candidate_feature_dict['Bandgap, eV'] = bandgap

    #try:
    electronegativity = calc_average_electronegativity(formula)
    #print('electronegativity: ',electronegativity)
    candidate_feature_dict['Average Mulliken electronegativity']=electronegativity

    valence_electrons = get_valence_electrons_number(formula)
    #print('valence_electrons: ',valence_electrons)
    candidate_feature_dict['Valence electrons']=valence_electrons

    oxygen_count = count_oxigen(formula)
    V = structure.volume
    Z = structure.composition.num_atoms / structure.composition.reduced_composition.num_atoms
    oxygen_conc = Z*oxygen_count/V
    candidate_feature_dict['Oxygen_concentration avg']=oxygen_conc

    packing_fraction = get_packing_fraction_from_formula_and_cell_volume(formula, V, Z)
    #print('packing_fraction: ',packing_fraction)
    candidate_feature_dict['Packing fraction avg']=packing_fraction
    candidate_feature_dict['Valence Electrons Density avg'] = valence_electrons/V

    print("candidate_feature_dict: ", candidate_feature_dict)
    print("Scaling...")
    X_before_scaler = np.zeros(len(scaler.feature_names_in_))
    #X_before_scaler = np.full(len(scaler.feature_names_in_), np.nan)
    #filling vectors
    for i in range(len(scaler.feature_names_in_)):
        f= scaler.feature_names_in_[i]
        if f in candidate_feature_dict:
            X_before_scaler[i]=candidate_feature_dict[f]
    #for i in range(len(scaler.feature_names_out_)):
    #    f= scaler.feature_names_out_[i] 
    #    if f not in candidate_feature_dict:
    #        print(f," not in candidate dictionary but is required by selector")
    print("len(scaler.feature_names_in_): ", len(scaler.feature_names_in_))
    print("scaler.feature_names_in_: ", scaler.feature_names_in_)
    print("X_before_scaler: ", X_before_scaler)
    X_after_scaler = scaler.transform(X_before_scaler.reshape(1, -1))
    print("X_after_scaler: ", X_after_scaler)
    X_after_selector = selector.transform(X_after_scaler)

    selected_features = selector.get_feature_names_out(scaler.feature_names_in_)
    print(f"The selector selected {len(selected_features)} features:")
    print(selected_features)
    for f in selected_features:
        if f not in candidate_feature_dict:
            print(f," is not in candidate dictionary but is required by selector")
            raise ValueError(f," is not in candidate dictionary but is required by selector")

    print("X_after_selector: ", X_after_selector)
    X_after_selector=X_after_selector[0]
    vec = X_after_selector
    print(vec)
    prediction = cat_boost_model.predict([vec])[0]
    print("Prediction: ", prediction)
    return prediction
    #except(Exception) as e:
    #    print("Error during prediction: ", e)
    #    return None

In [395]:
predict_candidate_from_cif("M_MP491", 4,True,30)

Predicting for:  M_MP491
loading cif locally from DATA
Structure is loaded
Nd2 Ti3 H2 O10
['avg s valence electrons', 'avg p valence electrons', 'avg d valence electrons', 'avg f valence electrons', 'frac s valence electrons', 'frac p valence electrons', 'frac d valence electrons', 'frac f valence electrons']
[np.float64(1.8823529411764706), np.float64(2.3529411764705883), np.float64(0.35294117647058826), np.float64(0.47058823529411764), np.float64(0.37209302325581395), np.float64(0.4651162790697675), np.float64(0.06976744186046513), np.float64(0.09302325581395349)]
{'Nd': 2.0, 'Ti': 3.0, 'H': 2.0, 'O': 10.0}
Nd2 Ti3 H2 O10
10.0
---------------------------------------------
Enter!
Nd2 Ti3 H2 O10
229.90248606119223
1.0
Element: Nd
Element ox state: 3
r:  1.123 ang
Nd   2.0
Element: Ti
Element ox state: 4
r:  0.745 ang
Ti   3.0
Element: H
Element ox state: 1
Local ionic radii table request for  H+
r:  0.02
H   2.0
Element: O
Element ox state: -2
r:  1.26 ang
O   10.0
candidate_feature_di

C:\Users\Nikita\AppData\Local\Temp\ipykernel_8248\3708692272.py:36: UserWarning: No ionic radius for H+!
  r = specie.ionic_radius
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RFE was fitted with feature names
  warnings.warn(


np.float64(8.629001720005386)

In [396]:
predict_candidate_from_cif("M_MP491", 4, True, np.nan)

Predicting for:  M_MP491
loading cif locally from DATA
Structure is loaded
Nd2 Ti3 H2 O10
d,A = NaN
['avg s valence electrons', 'avg p valence electrons', 'avg d valence electrons', 'avg f valence electrons', 'frac s valence electrons', 'frac p valence electrons', 'frac d valence electrons', 'frac f valence electrons']
[np.float64(1.8823529411764706), np.float64(2.3529411764705883), np.float64(0.35294117647058826), np.float64(0.47058823529411764), np.float64(0.37209302325581395), np.float64(0.4651162790697675), np.float64(0.06976744186046513), np.float64(0.09302325581395349)]
{'Nd': 2.0, 'Ti': 3.0, 'H': 2.0, 'O': 10.0}
Nd2 Ti3 H2 O10
10.0
---------------------------------------------
Enter!
Nd2 Ti3 H2 O10
229.90248606119223
1.0
Element: Nd
Element ox state: 3
r:  1.123 ang
Nd   2.0
Element: Ti
Element ox state: 4
r:  0.745 ang
Ti   3.0
Element: H
Element ox state: 1
Local ionic radii table request for  H+
r:  0.02
H   2.0
Element: O
Element ox state: -2
r:  1.26 ang
O   10.0
candidate_

C:\Users\Nikita\AppData\Local\Temp\ipykernel_8248\3708692272.py:36: UserWarning: No ionic radius for H+!
  r = specie.ionic_radius
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RFE was fitted with feature names
  warnings.warn(


np.float64(6.704713886121749)

# Inference

In [397]:
predict_candidate_from_cif("generated_cifs/gen_32_KLaO2.cif", 4, False)

Predicting for:  generated_cifs/gen_32_KLaO2.cif
loading cif locally


nan

In [398]:
def predict_candidate_bandgap_from_cif(cif):
    return predict_candidate_from_cif(cif,0)

In [399]:
def predict_candidate_log_rate_from_cif(row):
    cif = row['Path']
    bandgap = row['Bandgap_predicted']
    output = 0
    try:
        output = predict_candidate_from_cif(cif,bandgap, False)  #True - read from Data, False - read generated samples
    except:
        output = np.nan
    return output

In [400]:
s/0

NameError: name 's' is not defined

In [401]:
predicted_column = f"{title}_predicted"

if(title=="Log_rate"):
    df_candidates[predicted_column] = df_candidates.apply(predict_candidate_log_rate_from_cif,axis=1)
else:
    df_candidates[predicted_column] = df_candidates["Path"].apply(predict_candidate_bandgap_from_cif)

Predicting for:  generated_cifs\gen_10_Sr3CaZnO5.cif
loading cif locally
loadied cif locally
Structure is loaded
Sr3 Ca1 Zn1 O5
['avg s valence electrons', 'avg p valence electrons', 'avg d valence electrons', 'avg f valence electrons', 'frac s valence electrons', 'frac p valence electrons', 'frac d valence electrons', 'frac f valence electrons']
[np.float64(2.0), np.float64(2.0), np.float64(1.0), np.float64(0.0), np.float64(0.4), np.float64(0.4), np.float64(0.2), np.float64(0.0)]
{'Sr': 3.0, 'Ca': 1.0, 'Zn': 1.0, 'O': 5.0}
Sr3 Ca1 Zn1 O5
5.0
---------------------------------------------
Enter!
Sr3 Ca1 Zn1 O5
164.90543861304312
1.0
Element: Sr
Element ox state: 2
r:  1.32 ang
Sr   3.0
Element: Ca
Element ox state: 2
r:  1.14 ang
Ca   1.0
Element: Zn
Element ox state: 2
r:  0.88 ang
Zn   1.0
Element: O
Element ox state: -2
r:  1.26 ang
O   5.0
candidate_feature_dict:  {'CalcT(K)': 1296.237241, 'Calc time (h)': 19.567693, 'Promoter, w%': 0, 'Surface area, m2/g': 10.37065568513124, 'Alcoh

e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RFE was fitted with feature names
  warnings.warn(
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RFE was fitted with feature names
  warnings.warn(
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxSc

Prediction:  6.296880087332151
Predicting for:  generated_cifs\gen_17_Ca5O3.cif
loading cif locally
loadied cif locally
Structure is loaded
Ca5 O3
['avg s valence electrons', 'avg p valence electrons', 'avg d valence electrons', 'avg f valence electrons', 'frac s valence electrons', 'frac p valence electrons', 'frac d valence electrons', 'frac f valence electrons']
[np.float64(2.0), np.float64(1.5), np.float64(0.0), np.float64(0.0), np.float64(0.5714285714285714), np.float64(0.42857142857142855), np.float64(0.0), np.float64(0.0)]
{'Ca': 5.0, 'O': 3.0}
Ca5 O3
3.0
---------------------------------------------
Enter!
Ca5 O3
139.70997274919577
1.0
Element: Ca
Element ox state: 2
r:  1.14 ang
Ca   5.0
Element: O
Element ox state: -2
r:  1.26 ang
O   3.0
candidate_feature_dict:  {'CalcT(K)': 1296.237241, 'Calc time (h)': 19.567693, 'Promoter, w%': 0, 'Surface area, m2/g': 10.37065568513124, 'Alcohol, %': 1, 'Power, W': 1, 'Wave length min, nm': 200, 'Promotion method_PD': True, 'Promoter_Pt'

e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RFE was fitted with feature names
  warnings.warn(
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RFE was fitted with feature names
  warnings.warn(
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxSc

Prediction:  6.252391869399084
Predicting for:  generated_cifs\gen_23_Ca3O.cif
loading cif locally
loadied cif locally
Structure is loaded
Ca3 O1
['avg s valence electrons', 'avg p valence electrons', 'avg d valence electrons', 'avg f valence electrons', 'frac s valence electrons', 'frac p valence electrons', 'frac d valence electrons', 'frac f valence electrons']
[np.float64(2.0), np.float64(1.0), np.float64(0.0), np.float64(0.0), np.float64(0.6666666666666666), np.float64(0.3333333333333333), np.float64(0.0), np.float64(0.0)]
{'Ca': 3.0, 'O': 1.0}
Ca3 O1
1.0
---------------------------------------------
Enter!
Ca3 O1
101.8648477147743
1.0
Element: Ca
Element ox state: 2
r:  1.14 ang
Ca   3.0
Element: O
Element ox state: -2
r:  1.26 ang
O   1.0
candidate_feature_dict:  {'CalcT(K)': 1296.237241, 'Calc time (h)': 19.567693, 'Promoter, w%': 0, 'Surface area, m2/g': 10.37065568513124, 'Alcohol, %': 1, 'Power, W': 1, 'Wave length min, nm': 200, 'Promotion method_PD': True, 'Promoter_Pt': T

e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RFE was fitted with feature names
  warnings.warn(
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RFE was fitted with feature names
  warnings.warn(
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxSc

loadied cif locally
Structure is loaded
Sr2 Ca1 Nb1 O6
['avg s valence electrons', 'avg p valence electrons', 'avg d valence electrons', 'avg f valence electrons', 'frac s valence electrons', 'frac p valence electrons', 'frac d valence electrons', 'frac f valence electrons']
[np.float64(1.9), np.float64(2.4), np.float64(0.4), np.float64(0.0), np.float64(0.40425531914893614), np.float64(0.5106382978723404), np.float64(0.0851063829787234), np.float64(0.0)]
{'Sr': 2.0, 'Ca': 1.0, 'Nb': 1.0, 'O': 6.0}
Sr2 Ca1 Nb1 O6
6.0
---------------------------------------------
Enter!
Sr2 Ca1 Nb1 O6
146.9537559077694
1.0
Element: Sr
Element ox state: 2
r:  1.32 ang
Sr   2.0
Element: Ca
Element ox state: 2
r:  1.14 ang
Ca   1.0
Element: Nb
Element ox state: 5
r:  0.78 ang
Nb   1.0
Element: O
Element ox state: -2
r:  1.26 ang
O   6.0
candidate_feature_dict:  {'CalcT(K)': 1296.237241, 'Calc time (h)': 19.567693, 'Promoter, w%': 0, 'Surface area, m2/g': 10.37065568513124, 'Alcohol, %': 1, 'Power, W': 1, 'W

e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RFE was fitted with feature names
  warnings.warn(
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RFE was fitted with feature names
  warnings.warn(
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxSc

Prediction:  5.95936775364743
Predicting for:  generated_cifs\gen_7_CaO.cif
loading cif locally
loadied cif locally
Structure is loaded
Ca2 O2
['avg s valence electrons', 'avg p valence electrons', 'avg d valence electrons', 'avg f valence electrons', 'frac s valence electrons', 'frac p valence electrons', 'frac d valence electrons', 'frac f valence electrons']
[np.float64(2.0), np.float64(2.0), np.float64(0.0), np.float64(0.0), np.float64(0.5), np.float64(0.5), np.float64(0.0), np.float64(0.0)]
{'Ca': 2.0, 'O': 2.0}
Ca2 O2
2.0
---------------------------------------------
Enter!
Ca2 O2
57.9747180750398
2.0
Element: Ca
Element ox state: 2
r:  1.14 ang
Ca   2.0
Element: O
Element ox state: -2
r:  1.26 ang
O   2.0
candidate_feature_dict:  {'CalcT(K)': 1296.237241, 'Calc time (h)': 19.567693, 'Promoter, w%': 0, 'Surface area, m2/g': 10.37065568513124, 'Alcohol, %': 1, 'Power, W': 1, 'Wave length min, nm': 200, 'Promotion method_PD': True, 'Promoter_Pt': True, 'Promoter_Rh': False, 'Combin

e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RFE was fitted with feature names
  warnings.warn(


In [402]:
df_candidates.to_excel(f"{candidate_cif_folder}/candidates_after_inference_{title}.xlsx")

In [403]:
df_candidates

,Unnamed: 0,Formula,Composition,Path,avg s valence electrons,avg p valence electrons,avg d valence electrons,avg f valence electrons,frac s valence electrons,frac p valence electrons,frac d valence electrons,frac f valence electrons,Bandgap_predicted,Log_rate_predicted
0,0,Sr3 Ca1 Zn1 O5,Sr3 Ca1 Zn1 O5,generated_cifs\gen_10_Sr3CaZnO5.cif,2.000000,2.000000,1.000000,0.0,0.400000,0.400000,0.200000,0.000000,2.610747,7.910821
1,1,Ca3 Nb1 O3,Ca3 Nb1 O3,generated_cifs\gen_11_Ca3NbO3.cif,1.857143,1.714286,0.571429,0.0,0.448276,0.413793,0.137931,0.000000,2.125691,6.658949
2,2,Sr7 Ca1 O2,Sr7 Ca1 O2,generated_cifs\gen_12_Sr7CaO2.cif,2.000000,0.800000,0.000000,0.0,0.714286,0.285714,0.000000,0.000000,2.915892,6.725016
3,3,Ca6 P2 O2,Ca6 P2 O2,generated_cifs\gen_13_Ca3PO.cif,2.000000,1.400000,0.000000,0.0,0.588235,0.411765,0.000000,0.000000,2.840694,7.377565
4,4,Ca2 O2,Ca2 O2,generated_cifs\gen_14_CaO.cif,2.000000,2.000000,0.000000,0.0,0.500000,0.500000,0.000000,0.000000,3.424377,6.056471
5,5,Ca1 Hg1 O2,Ca1 Hg1 O2,generated_cifs\gen_15_CaHgO2.cif,2.000000,2.000000,2.500000,3.5,0.200000,0.200000,0.250000,0.350000,2.675027,4.516947
6,6,Ca4 O2,Ca4 O2,generated_cifs\gen_16_Ca2O.cif,2.000000,1.333333,0.000000,0.0,0.600000,0.400000,0.000000,0.000000,2.768946,6.296880
7,7,Ca5 O3,Ca5 O3,generated_cifs\gen_17_Ca5O3.cif,2.000000,1.500000,0.000000,0.0,0.571429,0.428571,0.000000,0.000000,2.911306,6.379108
8,8,Sr6 Ca1 Nb2 Zn1 O6,Sr6 Ca1 Nb2 Zn1 O6,generated_cifs\gen_18_Sr6CaNb2ZnO6.cif,1.875000,1.500000,1.125000,0.0,0.416667,0.333333,0.250000,0.000000,2.009304,7.545412
9,9,Sr5 Nb3 O7,Sr5 Nb3 O7,generated_cifs\gen_19_Sr5Nb3O7.cif,1.800000,1.866667,0.800000,0.0,0.402985,0.417910,0.179104,0.000000,2.297273,7.298883


In [404]:
import plotly.express as px

In [405]:
fig = px.histogram(
    df_candidates, 
    x=predicted_column, 
    nbins=20, 
    title="Value Distribution",
    width=500,
    height=500
)

fig.show()

In [407]:
clean_data = df_candidates[predicted_column].dropna()

# 2. Replicate Plotly's 20-bin histogram math exactly
counts, bin_edges = np.histogram(clean_data, bins=20,range=[0, 15] )

# 3. Calculate bin centers to match your exact column format
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

# 4. Construct your exact desired DataFrame layout
df_hist = pd.DataFrame({
    'Bin_Center': bin_centers,
    'Count': counts
})

print(df_hist)
if(title=="Log_rate"):
    df_hist.to_excel("Log_rate_hist.xlsx")


    Bin_Center  Count
0        0.375      0
1        1.125      0
2        1.875      0
3        2.625      0
4        3.375      0
5        4.125      0
6        4.875      1
7        5.625      7
8        6.375     14
9        7.125      5
10       7.875      5
11       8.625      0
12       9.375      0
13      10.125      0
14      10.875      0
15      11.625      0
16      12.375      0
17      13.125      0
18      13.875      0
19      14.625      0


In [ ]:
s/0

NameError: name 's' is not defined

# Create mattergen dataset

In [ ]:
import logging
import contextlib
import sys
from tqdm.auto import tqdm

In [ ]:
tqdm.pandas()

In [ ]:
def ID_to_formula(ID):
  formula = None
  try:
    structure = mpr.get_structure_by_material_id(ID)
    formula = structure.composition.formula
    return formula
  except:
    return None

In [ ]:
def is_oxide(MP_ID):
    #entry = None
    #try:
    #    entry = mpr.materials.summary.search(material_ids=[MP_ID],fields=["band_gap"])[0]
    #except:
    #    return False
    
    formula = ID_to_formula(MP_ID)
    if formula is None:
        return False
    comp=Composition(formula)
    return "O" in comp

In [ ]:
print(is_oxide("mp-626680"))
print(is_oxide("mp-1225695"))
print(is_oxide("mp-"))

Retrieving MaterialsDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

True


Retrieving MaterialsDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

False
False


In [ ]:
predict_candidate_from_cif("M_MP491",4,True)

Predicting for:  M_MP491
loading cif locally from DATA
Nd2 Ti3 H2 O10
['avg s valence electrons', 'avg p valence electrons', 'avg d valence electrons', 'avg f valence electrons', 'frac s valence electrons', 'frac p valence electrons', 'frac d valence electrons', 'frac f valence electrons']
[np.float64(1.8823529411764706), np.float64(2.3529411764705883), np.float64(0.35294117647058826), np.float64(0.47058823529411764), np.float64(0.37209302325581395), np.float64(0.4651162790697675), np.float64(0.06976744186046513), np.float64(0.09302325581395349)]
{'Nd': 2.0, 'Ti': 3.0, 'H': 2.0, 'O': 10.0}
Nd2 Ti3 H2 O10
10.0
---------------------------------------------
Enter!
Nd2 Ti3 H2 O10
229.90248606119223
1.0
Element: Nd
Element ox state: 3
r:  1.123 ang
Nd   2.0
Element: Ti
Element ox state: 4
r:  0.745 ang
Ti   3.0
Element: H
Element ox state: 1
Local ionic radii table request for  H+
r:  0.02
H   2.0
Element: O
Element ox state: -2
r:  1.26 ang
O   10.0
candidate_feature_dict:  {'CalcT(K)': 12

C:\Users\Nikita\AppData\Local\Temp\ipykernel_8248\3708692272.py:36: UserWarning: No ionic radius for H+!
  r = specie.ionic_radius
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RFE was fitted with feature names
  warnings.warn(


np.float64(6.704713886121749)

In [ ]:
import os
from pymatgen.io.cif import CifWriter
def get_cif_string_from_id(MP_ID):
  
  file_path="Data/CIF/" + str(MP_ID)+".cif"
  
  #print("Path: ",file_path)
  if os.path.exists(file_path):
    try:
      structure = Structure.from_file(file_path)
    except:
      print('ERROR: Invalid structure for ',MP_ID)
      return None
  else:
    return None

  if(structure == None):
    return None
  writer = CifWriter(structure)
  cif_string = str(writer)
  if not structure.is_ordered:
    return ""
  #print("CIF string: ",cif_string)
  return cif_string

In [ ]:
get_cif_string_from_id("M_MP140")

C:\Users\Nikita\AppData\Local\Temp\ipykernel_8248\1552821960.py:19: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['Ti0', 'Ti0', 'Ti1', 'Ti1', 'Ti2', 'Ti2', 'Bi3', 'Bi4', 'Bi5', 'Bi6', 'O7', 'O8', 'O9', 'O10', 'O11', 'O12', 'O13', 'O14', 'O15', 'O16', 'O17', 'O18']`.
  writer = CifWriter(structure)


''

In [ ]:
names = ["test_reduced","val_reduced","train_reduced"]
names = ["train_reduced"]
for name in names:
    df_mattergen = pd.read_excel(f"CIF_files_processing_output/soft 0_5/dataset_1.xlsx")
    print(df_mattergen.shape)
    df_mattergen.dropna(subset=["Bandgap, eV"], inplace=True)
    #df_mattergen = df_mattergen[df_mattergen["Rate, umol/(g*h)"] != 0]
    print(df_mattergen.shape)
    df_mattergen = df_mattergen.groupby('Perovskite', as_index=False).agg('first')
    df_mattergen["Log_rate"] = df_mattergen.progress_apply(lambda row: predict_candidate_from_cif(row["MP_CIF_modified"], row["Bandgap, eV"], True, row["d,A"]), axis=1)
    df_mattergen["cif"] = df_mattergen.progress_apply(lambda row: get_cif_string_from_id(row["MP_CIF_modified"]), axis=1)
    
    
    df_mattergen.to_csv(f"Data/Mattergen_dataset/with_activity/{name}_from_my_dataset.csv", index=False)

In [ ]:
s/0

In [ ]:
s/0

In [ ]:
names = ["test_reduced","val_reduced","train_reduced"]
names = ["train_reduced"]
for name in names:
    df_mattergen = pd.read_csv(f"Data/Mattergen_dataset/init/{name}.csv")
    print(df_mattergen.shape)
    df_mattergen.dropna(subset=["dft_band_gap"], inplace=True)
    df_mattergen = df_mattergen[df_mattergen["dft_band_gap"] != 0]
    print(df_mattergen.shape)
    #df_mattergen["Log_rate"]=df_mattergen["material_id"].progress_apply(lambda x: predict_activity_for_MP_ID(x))
    #df_mattergen.to_csv(f"Data/Mattergen_dataset/with_activity/{name}.csv", index=False)

    df_mattergen["Log_rate"] = df_mattergen.progress_apply(lambda row: predict_candidate_from_cif(row["material_id"], row["dft_band_gap"]), axis=1)
    
    
    df_mattergen.to_csv(f"Data/Mattergen_dataset/with_activity/{name}.csv", index=False)